# SCRESIA David DMLPA ablation — Codex definitive runner

This notebook is **not** the main thesis-decision paper lane. It is the David/DMLPA compatibility and architecture-ablation lane.

Use it to answer one narrow question: under the historical David handoff contract (`onehot_18d`, 19D `decision_reward`, 30-frame stack), does the DMLPA/Transformer extractor behave competitively?

Contract discipline:

- `onehot_18d` is preserved for compatibility with David's previous notebooks.
- The paper-clean thesis action contract is `thesis_factorized = MultiDiscrete([6,3])` and is tested in the ladder notebook.
- DMLPA v1.1 with positional encoding is allowed as a labeled architecture variant.
- Results from this notebook should be reported as an ablation, not as proof that the Transformer is the scientific contribution.


# Minimal Friend DMLPA notebook for SCRESIA

This is the final minimal David-lane notebook for Colab/Kaggle. It keeps the original Box(18) / 19D handoff contract for compatibility with the earlier notebooks, while making the model variant explicit.

Default run:

- action surface: `onehot_18d` / Box(18), kept as David compatibility, not the paper's strict thesis-decision main lane;
- observation surface: `decision_reward`, 19D before frame stack;
- temporal window: `VecFrameStack(30)`;
- model: DMLPA with sinusoidal positional encoding (`positional_v1_1`);
- algorithm: PPO by default for comparability with the repo baseline.

The original no-positional-encoding model remains available as `DMLPA_VARIANT = "original_v1_0"`. SAC remains available as `ALGORITHM = "SAC"`, but it is a different algorithmic lane and should not be mixed with PPO results without labeling it.


In [ ]:
# =====================
# 0) Minimal run config
# =====================

RUN_PROFILE = "debug"  # "debug" for smoke; "serious" for a full cloud run.

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

LOCAL_OUTPUT_DIR = "/content/friend_dmlpa_minimal_outputs"
KAGGLE_OUTPUT_DIR = "/kaggle/working/friend_dmlpa_minimal_outputs"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "torch>=2.1",
    "einops>=0.7",
    "pandas>=2.2",
]

SEED = 42
FRAME_STACK = 30
FEATURES_DIM = 120
DMLPA_VARIANT = "positional_v1_1"  # "original_v1_0" or "positional_v1_1"
ALGORITHM = "PPO"  # "PPO" for comparable runs; "SAC" for David's off-policy variant.
DEVICE = "auto"

REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
STOCHASTIC_PT = True
STEP_SIZE_HOURS = 168
ACTION_SPACE_MODE = "onehot_18d"
OBSERVATION_MODE = "decision_reward"

TOTAL_TIMESTEPS_DEBUG = 10_000
MAX_STEPS_DEBUG = 40
EVAL_EPISODES_DEBUG = 1

TOTAL_TIMESTEPS_SERIOUS = 100_000
MAX_STEPS_SERIOUS = 10_000
EVAL_EPISODES_SERIOUS = 5

N_STEPS_DEBUG = 128
BATCH_SIZE_DEBUG = 64
N_EPOCHS_DEBUG = 2
N_STEPS_SERIOUS = 1024
BATCH_SIZE_SERIOUS = 256
N_EPOCHS_SERIOUS = 10

if RUN_PROFILE == "debug":
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_DEBUG
    MAX_STEPS = MAX_STEPS_DEBUG
    EVAL_EPISODES = EVAL_EPISODES_DEBUG
    N_STEPS = N_STEPS_DEBUG
    BATCH_SIZE = BATCH_SIZE_DEBUG
    N_EPOCHS = N_EPOCHS_DEBUG
elif RUN_PROFILE == "serious":
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_SERIOUS
    MAX_STEPS = MAX_STEPS_SERIOUS
    EVAL_EPISODES = EVAL_EPISODES_SERIOUS
    N_STEPS = N_STEPS_SERIOUS
    BATCH_SIZE = BATCH_SIZE_SERIOUS
    N_EPOCHS = N_EPOCHS_SERIOUS
else:
    raise ValueError("RUN_PROFILE must be 'debug' or 'serious'.")

print({
    "RUN_PROFILE": RUN_PROFILE,
    "ALGORITHM": ALGORITHM,
    "DMLPA_VARIANT": DMLPA_VARIANT,
    "TOTAL_TIMESTEPS": TOTAL_TIMESTEPS,
    "MAX_STEPS": MAX_STEPS,
    "EVAL_EPISODES": EVAL_EPISODES,
    "N_STEPS": N_STEPS,
    "BATCH_SIZE": BATCH_SIZE,
    "N_EPOCHS": N_EPOCHS,
    "DEVICE": DEVICE,
})


In [ ]:
# =======================================
# 1) Portable Colab/Kaggle repository setup
# =======================================

import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.stderr:
        print(result.stderr[-2000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
else:
    print("local run: skipping pip install")

if IN_KAGGLE:
    REPO_DIR = Path(REPO_DIR_KAGGLE)
    output_root = Path(KAGGLE_OUTPUT_DIR)
elif IN_COLAB:
    REPO_DIR = Path(REPO_DIR_COLAB)
    output_root = Path(LOCAL_OUTPUT_DIR)
else:
    REPO_DIR = Path.cwd()
    output_root = Path.cwd() / "outputs" / "friend_dmlpa_minimal"

if IN_COLAB or IN_KAGGLE:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)])
    sys.path.insert(0, str(REPO_DIR.parent))
    sys.path.insert(0, str(REPO_DIR))
else:
    sys.path.insert(0, str(REPO_DIR.parent))
    sys.path.insert(0, str(REPO_DIR))

output_root.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR)
print("outputs:", output_root)
print("python path head:", sys.path[:3])


In [ ]:
# ==========================
# 2) Imports
# ==========================

import numpy as np
import pandas as pd
import torch
from einops import rearrange
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize

try:
    from scresia.supply_chain.external_env_interface import (
        get_episode_terminal_metrics,
        make_dkana_thesis_faithful_env,
    )
except ModuleNotFoundError:
    from supply_chain.external_env_interface import (
        get_episode_terminal_metrics,
        make_dkana_thesis_faithful_env,
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
print("sb3 device setting:", DEVICE)


In [ ]:
# ==============================================
# 3) Environment: David compatibility contract
# ==============================================

# This intentionally keeps David's default handoff contract:
# - action space: Box(18), raw realized thesis decision vector
# - observation before frame stack: 19D decision + reward
# - VecFrameStack(30), so DMLPA sees 30 frames
# This is not the paper's main thesis_factorized [6,3] lane.

def make_env(seed=SEED):
    def _init():
        env = make_dkana_thesis_faithful_env(
            reward_mode=REWARD_MODE,
            risk_level=RISK_LEVEL,
            stochastic_pt=STOCHASTIC_PT,
            action_space_mode=ACTION_SPACE_MODE,
            observation_mode=OBSERVATION_MODE,
            max_steps=MAX_STEPS,
            step_size_hours=STEP_SIZE_HOURS,
        )
        env.reset(seed=seed)
        return Monitor(env)
    return _init

venv = DummyVecEnv([make_env(SEED)])
env = VecFrameStack(venv, n_stack=FRAME_STACK)
env = VecNormalize(env, norm_reward=True, norm_obs=True)

obs = env.reset()
print("stacked observation shape:", obs.shape)
print("observation space:", env.observation_space)
print("action space:", env.action_space)

assert env.action_space.shape == (18,), env.action_space
assert env.observation_space.shape[0] % FRAME_STACK == 0, env.observation_space


## DMLPA architectures

`DMLPAOriginal` preserves the original no-positional-encoding architecture from `Untitled90.ipynb` and `notebookffc7b2c5ff.ipynb`.

`DMLPAPositionalV11` adds David's requested sinusoidal positional encoding and LayerNorm. This is a real architecture change, so it is labeled as v1.1 instead of being silently described as the original.


In [ ]:
class DMLPAOriginal(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor, features_dim=120):
        super().__init__(observation_space, features_dim)
        self.obs_dimension = observation_space.shape[0] // factor
        self.factor = factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        self.attlayer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim,
            nhead=12,
            batch_first=True,
        )
        self.accumulated = torch.nn.TransformerEncoder(self.attlayer, num_layers=4)

    def forward(self, observations):
        observations = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        observations = self.latent_rw(observations)
        observations = self.accumulated(observations)
        return observations[:, -1, :]


class DMLPAPositionalV11(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor, features_dim=120):
        super().__init__(observation_space, features_dim)
        self.obs_dimension = observation_space.shape[0] // factor
        self.factor = factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        self.attlayer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim,
            nhead=12,
            batch_first=True,
        )
        self.accumulated = torch.nn.TransformerEncoder(self.attlayer, num_layers=4)
        self.register_buffer("pos_encoding", self.build_sinusoidal_pe(factor, features_dim))

    @staticmethod
    def build_sinusoidal_pe(seq_len, d_model):
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, observations):
        observations = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        observations = self.latent_rw(observations)
        observations = observations + self.pos_encoding
        observations = self.pre_norm(observations)
        observations = self.accumulated(observations)
        return observations[:, -1, :]


DMLPA_BY_VARIANT = {
    "original_v1_0": DMLPAOriginal,
    "positional_v1_1": DMLPAPositionalV11,
}
DMLPA = DMLPA_BY_VARIANT[DMLPA_VARIANT]


In [ ]:
# ==============================================
# 4) Quick architecture shape check
# ==============================================

extractor = DMLPA(env.observation_space, factor=FRAME_STACK, features_dim=FEATURES_DIM)
with torch.no_grad():
    sample = torch.as_tensor(obs, dtype=torch.float32)
    features = extractor(sample)
print("DMLPA variant:", DMLPA_VARIANT)
print("DMLPA obs_dimension:", extractor.obs_dimension)
print("DMLPA feature shape:", tuple(features.shape))
assert features.shape[-1] == FEATURES_DIM


In [ ]:
# ==============================================
# 5) Train selected algorithm with DMLPA extractor
# ==============================================

policy_kwargs = dict(
    features_extractor_class=DMLPA,
    features_extractor_kwargs=dict(features_dim=FEATURES_DIM, factor=FRAME_STACK),
)

if ALGORITHM == "PPO":
    model = PPO(
        "MlpPolicy",
        env,
        policy_kwargs=policy_kwargs,
        device=DEVICE,
        verbose=1,
        n_steps=N_STEPS,
        batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS,
        seed=SEED,
    )
elif ALGORITHM == "SAC":
    model = SAC(
        "MlpPolicy",
        env,
        policy_kwargs=policy_kwargs,
        device=DEVICE,
        verbose=1,
        seed=SEED,
    )
else:
    raise ValueError("ALGORITHM must be 'PPO' or 'SAC'.")

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS)
elapsed_seconds = time.time() - start
print("training seconds:", elapsed_seconds)


In [ ]:
# ==============================================
# 6) Deterministic evaluation
# ==============================================

def get_base_env_from_vec(vec_env):
    current = vec_env
    while hasattr(current, "venv"):
        current = current.venv
    base = current.envs[0]
    while hasattr(base, "env"):
        base = base.env
    return base


def evaluate_model(model, n_episodes=EVAL_EPISODES):
    rows = []
    env.training = False
    env.norm_reward = False
    for episode in range(n_episodes):
        obs = env.reset()
        done = np.array([False])
        total_reward = 0.0
        steps = 0
        while not bool(done[0]):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, infos = env.step(action)
            total_reward += float(reward[0])
            steps += 1
        terminal = get_episode_terminal_metrics(get_base_env_from_vec(env))
        rows.append({
            "episode": episode,
            "steps": steps,
            "total_reward": total_reward,
            **terminal,
        })
    return pd.DataFrame(rows)

results = evaluate_model(model)
display(results)
print(results.mean(numeric_only=True))


In [ ]:
# ==============================================
# 7) Save artifacts
# ==============================================

run_name = f"{ALGORITHM.lower()}_friend_dmlpa_{DMLPA_VARIANT}"
model_path = output_root / f"{run_name}.zip"
vecnorm_path = output_root / f"{run_name}_vecnormalize.pkl"
results_path = output_root / f"{run_name}_evaluation_results.csv"
manifest_path = output_root / f"{run_name}_manifest.json"

model.save(str(model_path))
env.save(str(vecnorm_path))
results.to_csv(results_path, index=False)

manifest = {
    "notebook": "friend_dmlpa_minimal_colab_kaggle.ipynb",
    "run_profile": RUN_PROFILE,
    "algorithm": ALGORITHM,
    "dmlpa_variant": DMLPA_VARIANT,
    "seed": SEED,
    "frame_stack": FRAME_STACK,
    "features_dim": FEATURES_DIM,
    "action_space_mode": ACTION_SPACE_MODE,
    "observation_mode": OBSERVATION_MODE,
    "reward_mode": REWARD_MODE,
    "risk_level": RISK_LEVEL,
    "stochastic_pt": STOCHASTIC_PT,
    "step_size_hours": STEP_SIZE_HOURS,
    "max_steps": MAX_STEPS,
    "total_timesteps": TOTAL_TIMESTEPS,
    "eval_episodes": EVAL_EPISODES,
    "n_steps": N_STEPS if ALGORITHM == "PPO" else None,
    "batch_size": BATCH_SIZE if ALGORITHM == "PPO" else None,
    "n_epochs": N_EPOCHS if ALGORITHM == "PPO" else None,
    "device": DEVICE,
    "cuda_available": torch.cuda.is_available(),
    "training_seconds": elapsed_seconds,
    "model_path": str(model_path),
    "vecnormalize_path": str(vecnorm_path),
    "results_path": str(results_path),
    "claim_boundary": "David compatibility lane: Box(18)/19D handoff. Not the main thesis_factorized paper lane.",
}
manifest_path.write_text(json.dumps(manifest, indent=2))

print("saved model:", model_path)
print("saved vecnormalize:", vecnorm_path)
print("saved results:", results_path)
print("saved manifest:", manifest_path)
